# ts_regress_lag_y

> 对应代码：`EconometricsCode/code_in_notes/Chap.27/ts_regress_lag_y.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.27`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.27")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

与上一个文件相同的数据准备：读入季度 GDP、合并投资数据、声明时间序列并构造同比增长率。

In [ ]:
%%stata
clear
set more off

use ../datasets/quarterlyGDP.dta
merge 1:1 time using ../datasets/quarterlyInvest.dta

tsset time
gen gdp_growth=(gdp-L4.gdp)/L4.gdp*100

第一列：静态回归 `reg gdp_growth einvq0105`，随后用 `estat bgodfrey, lags(2)` 做 Breusch–Godfrey 检验（原假设为残差无 2 阶以内的自相关）。我们把检验的 p 值和当期投资系数（此时即长期效应）用 `outreg2 ... adds()` 写入结果表。

In [ ]:
%%stata
reg gdp_growth einvq0105
estat bgodfrey, lags(2)
local p=r(p)[1,1]
local le=_b[einvq0105]
outreg2 using ts_reg_lay_y.tex, replace adds(Breusch-Godfrey, `p', Long-run effects, `le')

第二列：加入被解释变量的一阶滞后与投资的当期及一阶滞后，得到 ADL(1,1) 型动态模型。对自回归分布滞后模型，投资对增长率的**长期效应**（long-run effect）为分子、分母分别汇总 X 与滞后 Y 的系数：$(\beta_0+\beta_1)/(1-\rho_1)$。我们用估计系数直接计算该长期乘数，并连同 BG 检验的 p 值一起 append 到结果表。

In [ ]:
%%stata
reg gdp_growth L.gdp_growth L(0/1).einvq0105
estat bgodfrey, lags(2)
local p=r(p)[1,1]
local le=(_b[einvq0105]+_b[L.einvq0105])/(1-_b[L.gdp_growth])
outreg2 using ts_reg_lay_y.tex, append adds(Breusch-Godfrey, `p', Long-run effects, `le')

第三列：被解释变量的滞后阶数增加到 2（ADL(2,1)），长期效应公式的分母相应变为 $1-\rho_1-\rho_2$。BG 检验用于检查残差自相关是否已被动态项充分吸收。

In [ ]:
%%stata
reg gdp_growth L(1/2).gdp_growth L(0/1).einvq0105
estat bgodfrey, lags(2)
local p=r(p)[1,1]
local le=(_b[einvq0105]+_b[L.einvq0105])/(1-_b[L.gdp_growth]-_b[L2.gdp_growth])
outreg2 using ts_reg_lay_y.tex, append adds(Breusch-Godfrey, `p', Long-run effects, `le')

第四列：在 ADL(2,1) 的基础上再增加投资的二阶滞后（ADL(2,2)），长期效应的分子也要加上 L2 期投资的系数。对比四列的 BG 检验 p 值与长期效应，可以评估动态设定是否充分、以及结论对模型设定的稳健性。

In [ ]:
%%stata
reg gdp_growth L(1/2).gdp_growth L(0/2).einvq0105
estat bgodfrey, lags(2)
local p=r(p)[1,1]
local le=(_b[einvq0105]+_b[L.einvq0105]+_b[L2.einvq0105])/(1-_b[L.gdp_growth]-_b[L2.gdp_growth])
outreg2 using ts_reg_lay_y.tex, append adds(Breusch-Godfrey, `p', Long-run effects, `le')